<a href="https://colab.research.google.com/github/Krispavnn555/NASSCOM_FDP/blob/main/DAY11_U24_Scalable_AI_Pipelines_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U24 — Scalable AI Pipelines: Lab

### Real-world brief: an MLOps pipeline for predictive maintenance

A factory wants to predict machine failures. Training a model is easy — the hard part is running it as a **reliable, reproducible, monitored system**. In this lab you'll build the MLOps scaffolding around a predictive-maintenance model: a **reusable feature pipeline**, an **experiment tracker**, a **model registry** with promote/rollback, a **training-serving skew** demonstration, **batch & online serving**, and **drift monitoring** that decides when to retrain — using a later 'production' batch where the machines have genuinely drifted.

**Resources provided:** `machine_health_train.csv` (labelled training data) and `machine_health_live.csv` (a later production batch with drift). Built with scikit-learn + joblib (no heavyweight MLOps platform needed — but every piece maps to MLflow / a model registry / a monitoring service).

_Phase F — ML Engineering / MLOps._

#objectives

Build a reusable feature pipeline (fit once, apply at train & serve)

Track experiments and pick the best run reproducibly

Register, version, promote and roll back models

Diagnose training-serving skew

Serve batch & online predictions, and monitor for data drift

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [1]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def _make_split(rng, n, drift=False):
    machine_type = rng.choice(["L", "M", "H"], n, p=[0.5, 0.3, 0.2])
    air_temp = rng.normal(298, 2, n)
    process_temp = air_temp + rng.normal(10, 1, n)
    rot_speed = rng.normal(1500, 150, n)
    torque = np.clip(rng.normal(40, 10, n), 4, None)
    tool_wear = rng.uniform(0, 250, n)

    if drift:
        # production drift: tools run longer & hotter than in training
        tool_wear = np.clip(tool_wear + rng.normal(60, 20, n), 0, 320)
        process_temp = process_temp + rng.normal(3.0, 0.5, n)
        torque = torque + rng.normal(4, 1, n)

    # failure: a sharp function of genuine stress drivers (learnable)
    drive = (0.015 * np.maximum(tool_wear - 180, 0) + 0.05 * np.maximum(torque - 50, 0)
             + 0.04 * np.maximum(process_temp - 312, 0)
             + 0.002 * np.abs(rot_speed - 1500))
    thr = np.quantile(drive, 0.80) if not drift else 0.42   # ~20% failure in train
    p = 1 / (1 + np.exp(-3.0 * (drive - thr)))
    failure = (rng.random(n) < p).astype(int)

    return pd.DataFrame({
        "machine_type": machine_type,
        "air_temp_k": air_temp.round(2),
        "process_temp_k": process_temp.round(2),
        "rot_speed_rpm": rot_speed.round(0),
        "torque_nm": torque.round(2),
        "tool_wear_min": tool_wear.round(1),
        "failure": failure,
    })


def build_pdm(train_path="machine_health_train.csv", live_path="machine_health_live.csv",
              seed=242, verbose=False):
    """Predictive-maintenance data for the MLOps lab (U24):

      machine_health_train.csv   labelled training data (~6000 rows)
      machine_health_live.csv    later 'production' batch (~2500 rows) with DRIFT injected
                                 (tools run longer/hotter) — for the drift-monitoring stage.
    """
    rng = np.random.default_rng(seed)
    train = _make_split(rng, 6000, drift=False)
    live = _make_split(rng, 2500, drift=True)
    train.to_csv(train_path, index=False)
    live.to_csv(live_path, index=False)
    if verbose:
        print("train:", train.shape, "| failure rate:", round(train.failure.mean(), 3))
        print("live :", live.shape, "| failure rate:", round(live.failure.mean(), 3))
        print("tool_wear mean  train vs live:", round(train.tool_wear_min.mean(), 1),
              "vs", round(live.tool_wear_min.mean(), 1), "(drift)")
    return train, live

if not (os.path.exists('machine_health_train.csv') and os.path.exists('machine_health_live.csv')):
    build_pdm(); print('Generated source files.')
else:
    print('Found the provided source files.')

Generated source files.


In [2]:
import pandas as pd, numpy as np, json, joblib, time
train = pd.read_csv('machine_health_train.csv')
live = pd.read_csv('machine_health_live.csv')
NUM = ['air_temp_k', 'process_temp_k', 'rot_speed_rpm', 'torque_nm', 'tool_wear_min']
CAT = ['machine_type']
TARGET = 'failure'
print('train:', train.shape, '| live:', live.shape)
print('train failure rate:', round(train.failure.mean(), 3))
train.head(3)

train: (6000, 7) | live: (2500, 7)
train failure rate: 0.329


,machine_type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,failure
0,L,301.23,309.40,1360.0,21.89,79.3,0
1,H,298.62,309.45,1498.0,43.76,118.5,0
2,L,301.74,312.20,1575.0,35.68,29.4,0


#1. A reusable feature pipeline

In [3]:
# -----------------------------------------------------------
# 🔹 1A. ONE transformer, fit on TRAIN, reused everywhere (no skew)
# -----------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

def make_pipeline(model):
    pre = ColumnTransformer([('num', StandardScaler(), NUM),
                             ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
    return Pipeline([('prep', pre), ('model', model)])

X = train[NUM + CAT]; y = train[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
pipe = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=0)).fit(Xtr, ytr)
pred = pipe.predict(Xte); proba = pipe.predict_proba(Xte)[:, 1]
print('baseline F1:', round(f1_score(yte, pred), 3), '| ROC-AUC:', round(roc_auc_score(yte, proba), 3))
print('The fitted Pipeline bundles preprocessing + model — the SAME transform at train and serve.')

baseline F1: 0.528 | ROC-AUC: 0.73
The fitted Pipeline bundles preprocessing + model — the SAME transform at train and serve.


#### 🧪 EXERCISE 1 — Add an engineered feature
1. Add a derived feature `temp_diff = process_temp_k - air_temp_k` (a known wear driver) to a copy of the data, include it in `NUM`, and retrain. Did F1 improve?
2. In a comment, explain why defining this feature *inside* the pipeline (so it's computed identically at serve time) avoids training-serving skew.

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

# 1. Add temp_diff, retrain, compare F1

# Custom transformer to add the temp_diff feature
class TempDiffFeature(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Ensure we're working on a copy to avoid modifying original dataframe unexpectedly
        X_copy = X.copy()
        X_copy['temp_diff'] = X_copy['process_temp_k'] - X_copy['air_temp_k']
        return X_copy

# Define the initial numerical features present in the 'train' DataFrame
# This list should NOT include 'temp_diff' as it's created by the transformer.
initial_numerical_features = ['air_temp_k', 'process_temp_k', 'rot_speed_rpm', 'torque_nm', 'tool_wear_min']

# Update the global NUM list to include the new feature 'temp_diff'
# This ensures the ColumnTransformer knows about it for scaling and OHE.
if 'temp_diff' not in NUM: # Prevent adding multiple times if cell is re-run
    NUM.append('temp_diff')

# Redefine the make_pipeline function to include the new feature transformer
def make_pipeline(model):
    pre = ColumnTransformer([('num', StandardScaler(), NUM),
                             ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
    # Add TempDiffFeature as the first step in the pipeline
    return Pipeline([
        ('temp_diff_feature', TempDiffFeature()),
        ('prep', pre),
        ('model', model)
    ])

# Retrain the model with the updated pipeline
# X is now created using only the features initially present in the 'train' DataFrame.
# The 'temp_diff_feature' step in the pipeline will add the 'temp_diff' column.
X = train[initial_numerical_features + CAT]; y = train[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

pipe_with_temp_diff = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=0)).fit(Xtr, ytr)
pred_td = pipe_with_temp_diff.predict(Xte); proba_td = pipe_with_temp_diff.predict_proba(Xte)[:, 1]

f1_temp_diff = round(f1_score(yte, pred_td), 3)
roc_auc_temp_diff = round(roc_auc_score(yte, proba_td), 3)

print(f'F1 with temp_diff: {f1_temp_diff} | ROC-AUC: {roc_auc_temp_diff}')
print('Baseline F1 (from previous cell): 0.528')

# 2. why compute features inside the pipeline:
# Defining features inside the pipeline ensures that the exact same transformation logic
# (e.g., calculating `temp_diff`) is applied consistently during both training and serving.
# This prevents training-serving skew, which occurs when features are computed differently
# in different environments, leading to discrepancies between model performance in development
# and production. By bundling the feature engineering within the fitted pipeline, we guarantee
# that the model always receives input in the expected format.

F1 with temp_diff: 0.532 | ROC-AUC: 0.732
Baseline F1 (from previous cell): 0.528


#2. Experiment tracking

In [7]:
# -----------------------------------------------------------
# 🔹 2A. A MINI EXPERIMENT TRACKER (params + metrics + artifact)
# -----------------------------------------------------------
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

runs = []
def log_run(name, model, params):
    p = make_pipeline(model).fit(Xtr, ytr)
    pr = p.predict(Xte); pb = p.predict_proba(Xte)[:, 1]
    rec = {'run': name, 'params': params,
           'f1': round(f1_score(yte, pr), 4), 'roc_auc': round(roc_auc_score(yte, pb), 4)}
    runs.append(rec); return p

log_run('rf_200',  RandomForestClassifier(n_estimators=200, random_state=0), {'n_estimators': 200})
log_run('gb_default', GradientBoostingClassifier(random_state=0), {'type': 'gradient_boosting'})
log_run('logreg', LogisticRegression(max_iter=1000), {'type': 'logistic'})
results = pd.DataFrame(runs).sort_values('roc_auc', ascending=False).reset_index(drop=True)
print(results[['run', 'f1', 'roc_auc']].to_string(index=False))
best_run = results.iloc[0]['run']
print('\nbest run by ROC-AUC:', best_run)

       run     f1  roc_auc
gb_default 0.5479   0.7566
    rf_200 0.5319   0.7324
    logreg 0.3598   0.6849

best run by ROC-AUC: gb_default


#### 🧪 EXERCISE 2 — Sweep & compare
1. Add three more runs sweeping RandomForest `max_depth` (e.g. 3, 6, 12). Log each.
2. Re-rank the tracker table and report whether a shallower or deeper forest wins here.
3. In a comment, explain why logging params+metrics+seed for every run is what makes results reproducible and comparable.

In [8]:
# 1-2. sweep max_depth, re-rank the runs table
log_run('rf_maxdepth_3', RandomForestClassifier(n_estimators=200, max_depth=3, random_state=0), {'n_estimators': 200, 'max_depth': 3})
log_run('rf_maxdepth_6', RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0), {'n_estimators': 200, 'max_depth': 6})
log_run('rf_maxdepth_12', RandomForestClassifier(n_estimators=200, max_depth=12, random_state=0), {'n_estimators': 200, 'max_depth': 12})

results = pd.DataFrame(runs).sort_values('roc_auc', ascending=False).reset_index(drop=True)
print(results[['run', 'f1', 'roc_auc']].to_string(index=False))
best_run = results.iloc[0]['run']
print('\nbest run by ROC-AUC:', best_run)

# 3. why track every run:
# Logging parameters, metrics, and random seeds for every run is crucial for reproducibility and comparability.
# Parameters: Records the exact configuration of the model, allowing others (or your future self) to recreate the experiment.
# Metrics: Provides a quantitative measure of performance, enabling objective comparison between different model configurations.
# Seed: Ensures that any stochastic elements (like initial weights or data splits) are consistent, making the results repeatable.
# Together, these elements form a complete record that allows for transparent analysis, debugging, and confident deployment of models.

           run     f1  roc_auc
 rf_maxdepth_6 0.5102   0.7589
    gb_default 0.5479   0.7566
 rf_maxdepth_3 0.4854   0.7558
rf_maxdepth_12 0.5191   0.7495
        rf_200 0.5319   0.7324
        logreg 0.3598   0.6849

best run by ROC-AUC: rf_maxdepth_6


#3. Model registry — version, promote, roll back

In [9]:
# -----------------------------------------------------------
# 🔹 3A. A FILE-BASED MODEL REGISTRY with stages
# -----------------------------------------------------------
os.makedirs('registry', exist_ok=True)
REG_FILE = 'registry/registry.json'
def load_registry():
    return json.load(open(REG_FILE)) if os.path.exists(REG_FILE) else {'models': []}
def register(pipe, name, metrics, stage='staging'):
    reg = load_registry()
    version = len(reg['models']) + 1
    path = f'registry/{name}_v{version}.joblib'
    joblib.dump(pipe, path)
    reg['models'].append({'name': name, 'version': version, 'stage': stage,
                          'path': path, 'metrics': metrics})
    json.dump(reg, open(REG_FILE, 'w'), indent=2)
    return version

# train & register the best model from tracking
best_model = RandomForestClassifier(n_estimators=200, random_state=0)
best_pipe = make_pipeline(best_model).fit(Xtr, ytr)
m = {'f1': round(f1_score(yte, best_pipe.predict(Xte)), 4)}
v = register(best_pipe, 'pdm_model', m, stage='staging')
print(f'registered pdm_model v{v} in staging with metrics {m}')

registered pdm_model v1 in staging with metrics {'f1': 0.5319}


In [10]:
# -----------------------------------------------------------
# 🔹 3B. PROMOTE to production (and keep rollback ability)
# -----------------------------------------------------------
def set_stage(name, version, stage):
    reg = load_registry()
    for mdl in reg['models']:
        if mdl['name'] == name and mdl['stage'] == 'production' and stage == 'production':
            mdl['stage'] = 'archived'        # demote the current prod model
        if mdl['name'] == name and mdl['version'] == version:
            mdl['stage'] = stage
    json.dump(reg, open(REG_FILE, 'w'), indent=2)
def production_model(name):
    reg = load_registry()
    cand = [m for m in reg['models'] if m['name'] == name and m['stage'] == 'production']
    return joblib.load(cand[-1]['path']) if cand else None

set_stage('pdm_model', v, 'production')
print('current registry stages:', [(m['version'], m['stage']) for m in load_registry()['models']])
print('production model loads:', production_model('pdm_model') is not None)

current registry stages: [(1, 'production')]
production model loads: True


#### 🧪 EXERCISE 3 — A bad deploy & a rollback
1. Register a **deliberately weak** v2 (e.g. `LogisticRegression` or a depth-1 tree) and promote it to production.
2. 'Discover' it's worse, then **roll back**: promote v1 back to production (your `set_stage` should archive v2). Confirm `production_model` now loads the good model again.
3. In a comment, explain why instant rollback to a known-good version is a core MLOps safety net.

In [11]:
# 1-2. register weak v2, promote, then roll back to v1
# Create and register a deliberately weak v2 model (e.g., LogisticRegression)
weak_model = LogisticRegression(random_state=0, max_iter=1000) # Ensure max_iter for convergence
weak_pipe = make_pipeline(weak_model).fit(Xtr, ytr)

# Calculate metrics for the weak model
pr_v2 = weak_pipe.predict(Xte)
pb_v2 = weak_pipe.predict_proba(Xte)[:, 1]
m_v2 = {'f1': round(f1_score(yte, pr_v2), 4), 'roc_auc': round(roc_auc_score(yte, pb_v2), 4)}

v2 = register(weak_pipe, 'pdm_model', m_v2, stage='staging')
print(f'Registered pdm_model v{v2} (weak model) in staging with metrics {m_v2}')

# Promote v2 to production
set_stage('pdm_model', v2, 'production')
print('\nAfter promoting v2 to production:')
print('Current registry stages:', [(m['version'], m['stage']) for m in load_registry()['models']])
print('Production model loads v2:', production_model('pdm_model') is not None)

# Roll back to v1 (promote v1 back to production)
set_stage('pdm_model', 1, 'production') # Assuming v1 was the first registered model
print('\nAfter rolling back to v1:')
print('Current registry stages:', [(m['version'], m['stage']) for m in load_registry()['models']])
prod_model_after_rollback = production_model('pdm_model')
print('Production model loads v1:', prod_model_after_rollback is not None)

# Confirming the F1 score of the rolled-back production model (v1)
# This requires knowing the metrics of v1, which can be retrieved from the registry
reg = load_registry()
v1_metrics = next(m for m in reg['models'] if m['name'] == 'pdm_model' and m['version'] == 1)['metrics']
print(f'Confirmed F1 of current production model (v1) is: {v1_metrics["f1"]}')

# 3. why rollback matters:
# Instant rollback to a known-good version is a core MLOps safety net because it allows for immediate recovery
# from faulty deployments. If a newly deployed model performs poorly, introduces bugs, or causes unexpected
# behavior in production, a quick rollback mechanism minimizes downtime and negative impact on users or business
# operations. It provides a crucial layer of resilience, allowing teams to quickly revert to a stable state
# while they debug and fix the issues with the problematic version offline.

Registered pdm_model v2 (weak model) in staging with metrics {'f1': 0.3598, 'roc_auc': np.float64(0.6849)}

After promoting v2 to production:
Current registry stages: [(1, 'archived'), (2, 'production')]
Production model loads v2: True

After rolling back to v1:
Current registry stages: [(1, 'production'), (2, 'archived')]
Production model loads v1: True
Confirmed F1 of current production model (v1) is: 0.5319


#4. Training-serving skew

In [13]:
# -----------------------------------------------------------
# 🔹 4A. THE #1 SILENT BUG: transforming serve data differently
# -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler
rf = RandomForestClassifier(n_estimators=200, random_state=0)

# Use the numerical features that are actually present in Xtr and live
# (before any pipeline-specific feature engineering like temp_diff)
numerical_features_for_this_cell = initial_numerical_features # defined in a previous cell

sc = StandardScaler().fit(Xtr[numerical_features_for_this_cell])              # scaler fitted on TRAIN
rf.fit(sc.transform(Xtr[numerical_features_for_this_cell]), ytr)
# serve on the later (drifted) production batch
Xlive_num = live[numerical_features_for_this_cell]; ylive = live[TARGET]
# RIGHT: reuse the TRAIN scaler -> drift signal (high tool_wear/temp) is preserved
f1_right = f1_score(ylive, rf.predict(sc.transform(Xlive_num)))
# WRONG: re-fit a fresh scaler on the serve batch -> it re-centres the drift away (skew!)
sc_wrong = StandardScaler().fit(Xlive_num)
f1_wrong = f1_score(ylive, rf.predict(sc_wrong.transform(Xlive_num)))
print(f'F1 reusing train scaler (correct): {f1_right:.3f}')
print(f'F1 re-fitting scaler at serve (skew): {f1_wrong:.3f}')
print('Same model, same rows — only the preprocessing differed, yet F1 collapsed.')
print('Re-fitting on serve data normalised the drift away, so the model never saw the danger signal.')

F1 reusing train scaler (correct): 0.763
F1 re-fitting scaler at serve (skew): 0.460
Same model, same rows — only the preprocessing differed, yet F1 collapsed.
Re-fitting on serve data normalised the drift away, so the model never saw the danger signal.


#### 🧪 EXERCISE 4 — Unseen category skew
1. The training-encoder used `handle_unknown='ignore'`. Create a serve row with a `machine_type` value never seen in training (e.g. `'X'`) and confirm the saved pipeline still predicts without crashing.
2. In a comment, explain why bundling the fitted encoder/scaler with the model (one artifact) is the clean fix for skew.

In [15]:
model = production_model('pdm_model') # Ensure the latest production model is loaded

# 1. predict on an unseen machine_type using the saved pipeline
# Create a new record with an unseen machine_type (e.g., 'X')
unseen_record = {
    'air_temp_k': 299,
    'process_temp_k': 313,
    'rot_speed_rpm': 1480,
    'torque_nm': 58,
    'tool_wear_min': 210,
    'machine_type': 'X' # This machine type was not in the training data
}

# Convert the record to a DataFrame, as expected by the pipeline
unseen_df = pd.DataFrame([unseen_record])

# Use the model (which contains the pipeline) to predict
try:
    # The pipeline handles feature engineering and one-hot encoding internally
    prediction_proba = model.predict_proba(unseen_df)[:, 1]
    print(f"Prediction for unseen machine_type 'X': {prediction_proba[0]:.3f}")
    print("Confirmed: The pipeline handles unseen categories without crashing due to 'handle_unknown=\'ignore\''.")
except Exception as e:
    print(f"An error occurred during prediction: {e}")
    print("This indicates the pipeline did not gracefully handle the unseen category.")

# 2. why ship transformer + model together:
# Bundling the fitted encoder/scaler (and any custom feature transformers) with the model into a single artifact
# (like a scikit-learn Pipeline) is the clean fix for training-serving skew. It guarantees that the exact
# same preprocessing steps and logic that were applied to the training data are also applied to the serving data.
# This prevents discrepancies that could arise from using different versions of preprocessing code,
# different parameter values for scalers/encoders, or subtle changes in feature engineering logic between
# training and deployment environments. It ensures consistency and maintains model performance in production.

Prediction for unseen machine_type 'X': 0.645
Confirmed: The pipeline handles unseen categories without crashing due to 'handle_unknown='ignore''.


#5. Serving — batch & online

In [17]:
# -----------------------------------------------------------
# 🔹 5A. ONLINE (one request) and BATCH (a file) inference
# -----------------------------------------------------------
model = production_model('pdm_model')         # load whatever is in production

# Define the raw input features as they appear in the demo record and live data
# This should NOT include 'temp_diff' as it's created by the pipeline.
RAW_INPUT_FEATURES = initial_numerical_features + CAT

def predict_online(record: dict):
    # The DataFrame should only contain the raw features that are provided directly
    row = pd.DataFrame([record])[RAW_INPUT_FEATURES]
    p = model.predict_proba(row)[0, 1]
    return {'failure_risk': round(float(p), 3), 'alert': bool(p > 0.5)}

demo = {'air_temp_k': 299, 'process_temp_k': 313, 'rot_speed_rpm': 1480,
        'torque_nm': 58, 'tool_wear_min': 210, 'machine_type': 'H'}
print('online prediction:', predict_online(demo))

def predict_batch(df):
    out = df.copy()
    # The DataFrame passed to the model should only contain the raw features
    out['failure_risk'] = model.predict_proba(df[RAW_INPUT_FEATURES])[:, 1]
    return out

scored = predict_batch(live.head(1000))
print('batch scored rows:', len(scored), '| flagged high-risk:', int((scored.failure_risk > 0.5).sum()))

online prediction: {'failure_risk': 0.635, 'alert': True}
batch scored rows: 1000 | flagged high-risk: 447


#### 🧪 EXERCISE 5 — Latency & a simple cache
1. Time 1000 calls to `predict_online`. Report average latency per call in milliseconds.
2. Add a tiny dict cache keyed on the rounded feature tuple so identical requests skip the model; show the speedup on a repeated request.
3. In a comment, name two other levers to cut online latency (from the U24 deck).

In [18]:
# 1-2. latency measurement + a prediction cache
import time

# Extend the predict_online function with a cache
prediction_cache = {}
def predict_online_with_cache(record: dict):
    # Create a cache key from relevant, rounded features
    # We'll use numerical features and machine_type, rounded to avoid floating point precision issues
    cache_key_features = ['air_temp_k', 'process_temp_k', 'rot_speed_rpm', 'torque_nm', 'tool_wear_min', 'machine_type']
    cache_key = tuple(round(record.get(f), 1) if isinstance(record.get(f), (int, float)) else record.get(f) for f in cache_key_features)

    if cache_key in prediction_cache:
        return prediction_cache[cache_key]

    # If not in cache, perform the actual prediction
    row = pd.DataFrame([record])[RAW_INPUT_FEATURES]
    p = model.predict_proba(row)[0, 1]
    result = {'failure_risk': round(float(p), 3), 'alert': bool(p > 0.5)}

    prediction_cache[cache_key] = result
    return result

# Time 1000 calls to predict_online (without cache initially for baseline)
num_calls = 1000

start_time = time.time()
for _ in range(num_calls):
    predict_online(demo)
end_time = time.time()
latency_no_cache = (end_time - start_time) / num_calls * 1000
print(f"Average latency per call (no cache, {num_calls} calls): {latency_no_cache:.3f} ms")

# Now, time with the cache, including a repeated request
# Clear cache for a fresh start
prediction_cache = {}

# First call (will populate cache)
start_time_cache_first = time.time()
first_prediction = predict_online_with_cache(demo)
end_time_cache_first = time.time()
latency_cache_first = (end_time_cache_first - start_time_cache_first) * 1000
print(f"\nLatency for first cached call (populates cache): {latency_cache_first:.3f} ms")

# Repeated call (will hit cache)
start_time_cache_repeat = time.time()
repeated_prediction = predict_online_with_cache(demo)
end_time_cache_repeat = time.time()
latency_cache_repeat = (end_time_cache_repeat - start_time_cache_repeat) * 1000
print(f"Latency for repeated cached call (cache hit): {latency_cache_repeat:.3f} ms")
print(f"Cache hit for demo record: {first_prediction}")

# 3. other latency levers:
# Other levers to cut online latency include:
# - Model Optimization: Using smaller, faster models (e.g., distillation, quantization) or converting models to a more efficient format (e.g., ONNX).
# - Infrastructure Optimization: Deploying models closer to users (edge computing), using more powerful hardware (GPUs/TPUs), or optimizing network infrastructure.

Average latency per call (no cache, 1000 calls): 23.402 ms

Latency for first cached call (populates cache): 20.663 ms
Latency for repeated cached call (cache hit): 0.076 ms
Cache hit for demo record: {'failure_risk': 0.635, 'alert': True}


#6. Drift monitoring & retraining

In [20]:
# -----------------------------------------------------------
# 🔹 6A. DATA DRIFT via Population Stability Index (PSI)
# -----------------------------------------------------------
def psi(expected, actual, bins=10):
    qs = np.quantile(expected, np.linspace(0, 1, bins + 1))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.histogram(expected, qs)[0] / len(expected) + 1e-6
    a = np.histogram(actual, qs)[0] / len(actual) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))

print('PSI (train vs live production batch) — >0.2 = significant drift:')
# Use 'initial_numerical_features' as 'temp_diff' is created by the pipeline and not in raw dataframes
drift = {f: round(psi(train[f], live[f]), 3) for f in initial_numerical_features}
for f, v in sorted(drift.items(), key=lambda x: -x[1]):
    flag = 'DRIFT' if v > 0.2 else 'ok'
    print(f'  {f:18s} PSI={v:.3f}  [{flag}]')
drifted = [f for f, v in drift.items() if v > 0.2]
print('\nfeatures that drifted:', drifted)

PSI (train vs live production batch) — >0.2 = significant drift:
  process_temp_k     PSI=1.465  [DRIFT]
  tool_wear_min      PSI=0.959  [DRIFT]
  torque_nm          PSI=0.140  [ok]
  air_temp_k         PSI=0.007  [ok]
  rot_speed_rpm      PSI=0.004  [ok]

features that drifted: ['process_temp_k', 'tool_wear_min']


In [22]:
Xl, yl = live[initial_numerical_features + CAT], live[TARGET]
f1_stale = f1_score(yl, model.predict(Xl))                 # current prod model on drifted data
# continuous training: retrain including recent production data
Xall = pd.concat([train[initial_numerical_features + CAT], live[initial_numerical_features + CAT].iloc[:1500]])
yall = pd.concat([train[TARGET], live[TARGET].iloc[:1500]])
retrained = make_pipeline(RandomForestClassifier(n_estimators=200, random_state=0)).fit(Xall, yall)
f1_retrained = f1_score(yl.iloc[1500:], retrained.predict(Xl.iloc[1500:]))
f1_old_holdout = f1_score(yl.iloc[1500:], model.predict(Xl.iloc[1500:]))
print(f'on drifted production data — stale model F1: {f1_old_holdout:.3f}')
print(f'after retraining on recent data  F1: {f1_retrained:.3f}')
print('Monitoring caught the drift; continuous training restored performance.')

on drifted production data — stale model F1: 0.731
after retraining on recent data  F1: 0.801
Monitoring caught the drift; continuous training restored performance.


#### 🧪 EXERCISE 6 — Close the loop
1. If the retrained model is better on recent data, **register it as v-next and promote to production** (reuse your registry functions). Confirm `production_model` now returns the new one.
2. In a comment, sketch the automated MLOps loop this completes: monitor → detect drift → retrain → validate → promote (with a human gate where appropriate).

In [ ]:
# 1. register & promote the retrained model
# YOUR CODE HERE

# 2. the closed MLOps loop: ...   (comment)

In [23]:
model = production_model('pdm_model')

# 1. register & promote the retrained model

# Calculate metrics for the retrained model on the holdout data
pr_retrained = retrained.predict(Xte)
pb_retrained = retrained.predict_proba(Xte)[:, 1]
m_retrained = {'f1': round(f1_score(yte, pr_retrained), 4), 'roc_auc': round(roc_auc_score(yte, pb_retrained), 4)}

# Register the retrained model
v_new = register(retrained, 'pdm_model', m_retrained, stage='staging')
print(f'Registered pdm_model v{v_new} (retrained model) in staging with metrics {m_retrained}')

# Promote the retrained model (v_new) to production
set_stage('pdm_model', v_new, 'production')
print(f'Promoted pdm_model v{v_new} to production.')

# Confirm that production_model now loads the new one
current_prod_model = production_model('pdm_model')
current_prod_version = [m['version'] for m in load_registry()['models'] if m['name'] == 'pdm_model' and m['stage'] == 'production'][0]
print(f'Confirmed: Current production model is now v{current_prod_version}.')
print(f'Its F1 score on holdout data is: {m_retrained["f1"]}')


Registered pdm_model v3 (retrained model) in staging with metrics {'f1': 1.0, 'roc_auc': np.float64(1.0)}
Promoted pdm_model v3 to production.
Confirmed: Current production model is now v3.
Its F1 score on holdout data is: 1.0


# 2. The closed MLOps loop:

### Automated MLOps Loop

This exercise completes a crucial automated MLOps loop:

1.  **Monitor:** Continuously observe model performance, data drift, and other operational metrics in production.
2.  **Detect Drift:** Identify significant changes in incoming data distributions (e.g., using PSI) compared to training data. When drift exceeds a predefined threshold, it triggers the next step.
3.  **Retrain:** If drift is detected or performance degrades, the system automatically initiates a retraining process using new, relevant data (often including the recent production data).
4.  **Validate:** The newly retrained model is rigorously evaluated (e.g., on a holdout set or by A/B testing) to ensure it performs better than the existing production model and meets performance criteria.
5.  **Promote:** If the retrained model passes validation, it is promoted to production (possibly after a human review/gate, especially for critical systems), replacing the older version. The previous version is often archived or demoted, but remains available for quick rollback.

This cyclical process ensures that models remain relevant and performant in dynamic real-world environments, automatically adapting to changes in data over time.

#📘 Summary

| Stage | What you built |
| ----- | -------------- |
| Feature pipeline | one fitted transformer reused at train & serve |
| Experiment tracking | logged params+metrics; picked the best run |
| Model registry | versioned models with stages, promote & rollback |
| Skew | proved why the transformer must be persisted, not re-fit |
| Serving | online (one request) and batch (a file) inference |
| Monitoring | PSI drift detection → continuous retraining |

**Core lesson:** the model is a small piece — the system around it (reproducible pipelines, tracked experiments, a registry, skew-free serving, and drift monitoring that triggers retraining) is what keeps ML working in production as the world changes.

**Next — U25:** a different learning paradigm entirely — agents that learn from reward.